# Credit Card Transaction Analysis (Python)
**Data:** simulated credit card transactions from Kaggle (`kartik2112/fraud-detection`, generated with the Sparkov tool). This is **not** real customer data and **not** American Express data.

**Scope notes**
- Uses `fraudTrain.csv`. The date range is computed below, not assumed.
- "Customer" = card. The raw card number (`cc_num`) is replaced by a non-reversible surrogate `card_id` (C0001, C0002, ...) right after loading, and is dropped from memory and from every export.
- `is_fraud` is a simulator label. Findings are descriptive associations, **not** a fraud detection model and not proof of cause.

**Sections:** 1 Setup, 2 Load, 3 Inspect, 4 Clean, 5 Features, 6 KPIs, 7 Amounts, 8 Categories, 9 Time trends, 10 Fraud, 11 Customers, 12 Outliers, 13 Geography, 14 Export, 15 Facts summary

## 1. Setup

In [ ]:
import os, sys, warnings
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
pd.options.display.float_format = "{:,.2f}".format
pd.options.display.max_columns = 50
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["axes.titleweight"] = "bold"
RNG = 42  # fixed seed for any sampling -> reproducible

print(f"python {sys.version.split()[0]} | pandas {pd.__version__} | numpy {np.__version__} | "
      f"matplotlib {matplotlib.__version__} | seaborn {sns.__version__}")

## 2. Load the data
In Colab this downloads the dataset with `kagglehub`. To use a file you already have, set the `DATA_FILE` environment variable (or edit the line below) to its path.

Memory: only the columns needed are read (names and street are never loaded), text columns use the `category` dtype, and the file has about 1.3M rows.

A **schema check** confirms the CSV header matches what this notebook (and the SQL scripts) expect, so a changed file fails loudly instead of silently giving wrong numbers.

In [ ]:
DATA_FILE = os.environ.get("DATA_FILE", "")   # optional: path to fraudTrain.csv

if not DATA_FILE:
    try:
        import kagglehub
    except ImportError:
        import subprocess
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "kagglehub"])
        import kagglehub
    DATA_FILE = os.path.join(kagglehub.dataset_download("kartik2112/fraud-detection"), "fraudTrain.csv")
print("Reading:", DATA_FILE)

# Schema check: column 0 is an unnamed row index; the other 22 columns must match in name AND order.
EXPECTED = ["trans_date_trans_time", "cc_num", "merchant", "category", "amt", "first", "last", "gender",
            "street", "city", "state", "zip", "lat", "long", "city_pop", "job", "dob", "trans_num",
            "unix_time", "merch_lat", "merch_long", "is_fraud"]
header = pd.read_csv(DATA_FILE, nrows=0).columns.tolist()
print("First header cell:", repr(header[0]), "| total header cells:", len(header))
assert header[1:] == EXPECTED, f"Unexpected columns/order: {header}"

use_cols = ["trans_date_trans_time", "cc_num", "merchant", "category", "amt", "gender",
            "city", "state", "zip", "lat", "long", "city_pop", "job", "dob", "trans_num",
            "merch_lat", "merch_long", "is_fraud"]
dtypes = {"cc_num": "int64", "merchant": "category", "category": "category", "amt": "float64",
          "gender": "category", "city": "category", "state": "category", "zip": "int32",
          "lat": "float64", "long": "float64", "city_pop": "int32", "job": "category",
          "merch_lat": "float64", "merch_long": "float64", "is_fraud": "int64"}
# float64 for coordinates keeps distances identical to the SQL (FLOAT) calculation; is_fraud is int64
# because small integer types overflow when counts are multiplied by 100 for percentages.

df = pd.read_csv(DATA_FILE, usecols=use_cols, dtype=dtypes)
print(f"Loaded {len(df):,} rows x {df.shape[1]} columns | memory: {df.memory_usage(deep=True).sum()/1e6:,.0f} MB")

## 3. Replace the card number with a safe customer ID
`card_id` is the rank of the card number among all distinct card numbers (C0001, C0002, ...). It carries no digits of the original number. The SQL scripts build the same IDs, so results can be compared across tools.
(The data is simulated. On real data you would use a secret-keyed hash instead, because a public rank order could be re-derived by someone who holds the full card list.)

In [ ]:
card_numbers = np.sort(df["cc_num"].unique())
width = max(4, len(str(len(card_numbers))))
card_labels = [f"C{i:0{width}d}" for i in range(1, len(card_numbers) + 1)]
codes = np.searchsorted(card_numbers, df["cc_num"].to_numpy())
df["card_id"] = pd.Categorical.from_codes(codes, categories=card_labels)
n_cards = len(card_numbers)
df = df.drop(columns="cc_num")                    # raw card number is no longer in memory
del card_numbers, codes
print(f"{n_cards:,} distinct cards -> IDs {card_labels[0]} ... {card_labels[-1]}")

## 4. Inspect

In [ ]:
display(df.head(3).T)
df.info(memory_usage="deep")

## 5. Clean: types, missing values, duplicates, invalid values
Checks are computed from the data. A fix is applied only where a problem is found (and without copying the whole table when nothing needs fixing).

In [ ]:
# 5.1 Data types: parse timestamps and birth dates
df["trans_date_trans_time"] = pd.to_datetime(df["trans_date_trans_time"], errors="coerce")
df["dob"] = pd.to_datetime(df["dob"], errors="coerce")
print("Unparseable timestamps:", int(df["trans_date_trans_time"].isna().sum()),
      "| unparseable birth dates:", int(df["dob"].isna().sum()))

# 5.2 Missing values
missing = df.isna().sum()
print("\nColumns with missing values:")
print(missing[missing > 0] if (missing > 0).any() else "None")

# 5.3 Duplicates (full row and transaction ID)
n_dup_rows = int(df.duplicated().sum())
n_dup_ids = int(df["trans_num"].duplicated().sum())
print(f"\nDuplicate rows: {n_dup_rows:,} | duplicate trans_num: {n_dup_ids:,}")

# 5.4 Invalid values
bad_amt = df["amt"] <= 0
bad_flag = ~df["is_fraud"].isin([0, 1])
invalid = {
    "amt <= 0": int(bad_amt.sum()),
    "is_fraud not in {0,1}": int(bad_flag.sum()),
    "customer lat/long out of range": int((~df["lat"].between(-90, 90) | ~df["long"].between(-180, 180)).sum()),
    "merchant lat/long out of range": int((~df["merch_lat"].between(-90, 90) | ~df["merch_long"].between(-180, 180)).sum()),
    "dob after transaction": int((df["dob"] > df["trans_date_trans_time"]).sum()),
}
print("\nInvalid value checks:")
for k, v in invalid.items():
    print(f"  {k}: {v:,}")

# 5.5 Treatment, only when a problem exists
before = len(df)
if n_dup_ids:
    df = df.drop_duplicates(subset="trans_num", keep="first")
if n_dup_rows:
    df = df.drop_duplicates()
drop_mask = df[["trans_date_trans_time", "amt", "is_fraud"]].isna().any(axis=1) | (df["amt"] <= 0) | ~df["is_fraud"].isin([0, 1])
if drop_mask.any():
    df = df[~drop_mask].copy()
print(f"\nRows removed by cleaning: {before - len(df):,} | rows kept: {len(df):,}")

**Merchant names:** the simulator adds a `fraud_` prefix to merchant names. It is a naming artefact, not a fraud signal, so it is removed only if present. This is done on the unique merchant names (not on 1.3M rows) to save memory.

In [ ]:
m_cats = df["merchant"].cat.categories
has_prefix = m_cats.str.startswith("fraud_")
prefix_share = has_prefix[df["merchant"].cat.codes.to_numpy()].mean()
print(f"Share of rows whose merchant starts with 'fraud_': {prefix_share:.1%}")

clean_names = m_cats.str.replace(r"^fraud_", "", regex=True)
new_cats = pd.Index(pd.unique(clean_names))
new_codes = new_cats.get_indexer(clean_names)[df["merchant"].cat.codes.to_numpy()]
df["merchant_clean"] = pd.Categorical.from_codes(new_codes, categories=new_cats)
df = df.drop(columns="merchant")
del m_cats, has_prefix, clean_names, new_codes
print("Distinct merchants after cleaning:", df["merchant_clean"].nunique())

## 6. Date and time features

In [ ]:
ts = df["trans_date_trans_time"]
df["txn_date"] = ts.dt.normalize()
df["txn_month"] = ts.dt.to_period("M").dt.to_timestamp()
df["txn_hour"] = ts.dt.hour.astype("int8")
df["txn_weekday"] = pd.Categorical(ts.dt.day_name(),
    categories=["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"], ordered=True)

# Exact age in whole years at the transaction date (same rule as the SQL view)
had_birthday = (ts.dt.month > df["dob"].dt.month) | ((ts.dt.month == df["dob"].dt.month) & (ts.dt.day >= df["dob"].dt.day))
df["customer_age"] = (ts.dt.year - df["dob"].dt.year - (~had_birthday).astype(int)).astype("int16")

# Amount bands: same edges and labels as the SQL view (lower edge included, upper edge excluded)
BAND_LABELS = ["1. Under 10", "2. 10-49.99", "3. 50-99.99", "4. 100-249.99", "5. 250-499.99", "6. 500-999.99", "7. 1000 and above"]
df["amount_band"] = pd.cut(df["amt"], [0, 10, 50, 100, 250, 500, 1000, np.inf], labels=BAND_LABELS, right=False)

# Haversine distance (km) between customer and merchant (float64 coordinates, same formula as SQL)
lat1, lon1, lat2, lon2 = (np.radians(df[c].to_numpy()) for c in ["lat", "long", "merch_lat", "merch_long"])
a = np.sin((lat2 - lat1) / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2
df["distance_km"] = 2 * 6371 * np.arcsin(np.sqrt(np.clip(a, 0, 1)))
del lat1, lon1, lat2, lon2, a

df["fraud_label"] = pd.Categorical(np.where(df["is_fraud"] == 1, "Fraud", "Legitimate"), categories=["Legitimate", "Fraud"])

print("Date range:", ts.min(), "->", ts.max())
last_month = df["txn_month"].max()
last_month_days = df.loc[df["txn_month"] == last_month, "txn_date"].nunique()
print(f"Last month in data ({last_month:%Y-%m}) has {last_month_days} distinct days -> may be a partial month")

## 7. Headline KPIs

In [ ]:
total_txn = len(df)
total_amt = df["amt"].sum()
n_fraud = int(df["is_fraud"].sum())
kpis = pd.Series({
    "Total transactions": total_txn,
    "Total amount (USD)": total_amt,
    "Average amount (USD)": df["amt"].mean(),
    "Median amount (USD)": df["amt"].median(),
    "Unique cards": df["card_id"].nunique(),
    "Unique merchants": df["merchant_clean"].nunique(),
    "Fraud-labelled transactions": n_fraud,
    "Fraud rate (% of transactions)": 100 * n_fraud / total_txn,
    "Fraud amount (USD)": df.loc[df["is_fraud"] == 1, "amt"].sum(),
    "Fraud amount (% of total amount)": 100 * df.loc[df["is_fraud"] == 1, "amt"].sum() / total_amt,
})
display(kpis.to_frame("value"))

## 8. Transaction amount distribution

In [ ]:
display(df["amt"].describe(percentiles=[.25, .5, .75, .9, .95, .99]).to_frame("amt"))

fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
sns.histplot(df["amt"], bins=80, ax=ax[0], color="#1f77b4")
ax[0].set_title("Transaction amount (raw scale: heavily right-skewed)")
ax[0].set_xlabel("Amount (USD)")
sns.histplot(np.log10(df["amt"]), bins=80, ax=ax[1], color="#1f77b4")
ax[1].set_title("Transaction amount (log10 scale)")
ax[1].set_xlabel("log10(Amount in USD)")
plt.tight_layout(); plt.show()

## 9. Category analysis

In [ ]:
cat = (df.groupby("category", observed=True)
         .agg(txns=("amt", "size"), total_amt=("amt", "sum"), avg_amt=("amt", "mean"),
              fraud_count=("is_fraud", "sum"))
         .assign(pct_of_spend=lambda x: 100 * x["total_amt"] / x["total_amt"].sum(),
                 fraud_rate_pct=lambda x: 100 * x["fraud_count"] / x["txns"])
         .sort_values("total_amt", ascending=False))
display(cat)

fig, ax = plt.subplots(1, 2, figsize=(14, 5))
sns.barplot(y=cat.index.astype(str), x=cat["total_amt"] / 1e6, ax=ax[0], color="#1f77b4")
ax[0].set_title("Total spend by category"); ax[0].set_xlabel("USD millions"); ax[0].set_ylabel("")
sns.barplot(y=cat.index.astype(str), x=cat["avg_amt"], ax=ax[1], color="#ff7f0e")
ax[1].set_title("Average transaction amount by category"); ax[1].set_xlabel("USD"); ax[1].set_ylabel("")
plt.tight_layout(); plt.show()

top_m = (df.groupby("merchant_clean", observed=True)
           .agg(txns=("amt", "size"), total_amt=("amt", "sum")).nlargest(10, "txns"))
display(top_m)
fig, ax = plt.subplots(figsize=(9, 4.5))
sns.barplot(y=top_m.index.astype(str), x=top_m["txns"], ax=ax, color="#2ca02c")
ax.set_title("Top 10 merchants by transaction count"); ax.set_xlabel("Transactions"); ax.set_ylabel("")
plt.tight_layout(); plt.show()

## 10. Time patterns

In [ ]:
monthly = (df.groupby("txn_month")
             .agg(txns=("amt", "size"), total_amt=("amt", "sum"), avg_amt=("amt", "mean"),
                  frauds=("is_fraud", "sum"))
             .assign(fraud_rate_pct=lambda x: 100 * x["frauds"] / x["txns"]))
monthly["mom_spend_change_pct"] = monthly["total_amt"].pct_change() * 100
display(monthly)

fig, ax = plt.subplots(figsize=(12, 4.5))
ax.plot(monthly.index, monthly["total_amt"] / 1e6, marker="o", color="#1f77b4")
ax.set_title("Monthly transaction amount (last month may be partial)")
ax.set_ylabel("USD millions"); ax.set_xlabel("")
plt.tight_layout(); plt.show()

weekday = (df.groupby("txn_weekday", observed=True)
             .agg(txns=("amt", "size"), total_amt=("amt", "sum"), avg_amt=("amt", "mean"), frauds=("is_fraud", "sum"))
             .assign(fraud_rate_pct=lambda x: 100 * x["frauds"] / x["txns"]))
display(weekday)

hw = df.groupby(["txn_weekday", "txn_hour"], observed=True).size().unstack(fill_value=0)
fig, ax = plt.subplots(figsize=(13, 4.5))
sns.heatmap(hw, cmap="Blues", ax=ax, cbar_kws={"label": "Transactions"})
ax.set_title("Transactions by weekday and hour"); ax.set_xlabel("Hour of day"); ax.set_ylabel("")
plt.tight_layout(); plt.show()

## 11. Fraud analysis
Labels come from the simulator. Rates below describe the labelled data; they say nothing about causes.

In [ ]:
summary = (df.groupby("fraud_label", observed=True)
             .agg(txns=("amt", "size"), total_amt=("amt", "sum"), avg_amt=("amt", "mean"),
                  median_amt=("amt", "median")))
summary["pct_of_txns"] = 100 * summary["txns"] / summary["txns"].sum()
summary["pct_of_amt"] = 100 * summary["total_amt"] / summary["total_amt"].sum()
display(summary)

sample = df.sample(min(200_000, len(df)), random_state=RNG)   # sample only for box plots
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
sns.barplot(x=summary.index.astype(str), y=summary["txns"], ax=ax[0], palette=["#1f77b4", "#d62728"])
ax[0].set_yscale("log"); ax[0].set_title("Transactions by label (log scale)"); ax[0].set_xlabel("")
sns.boxplot(data=sample, x="fraud_label", y="amt", ax=ax[1],
            palette=["#1f77b4", "#d62728"], showfliers=False)
ax[1].set_title("Amount by label (outliers hidden; 200k-row sample)"); ax[1].set_xlabel("")
plt.tight_layout(); plt.show()

In [ ]:
fcat = cat.sort_values("fraud_rate_pct", ascending=False)
fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(y=fcat.index.astype(str), x=fcat["fraud_rate_pct"], ax=ax, color="#d62728")
ax.axvline(100 * n_fraud / total_txn, color="black", ls="--", label="Overall rate")
ax.set_title("Fraud rate by category"); ax.set_xlabel("% of transactions labelled fraud"); ax.set_ylabel("")
ax.legend(); plt.tight_layout(); plt.show()

hourly = (df.groupby("txn_hour").agg(txns=("amt", "size"), frauds=("is_fraud", "sum"))
            .assign(fraud_rate_pct=lambda x: 100 * x["frauds"] / x["txns"]))
fig, ax = plt.subplots(figsize=(11, 4.5))
sns.barplot(x=hourly.index, y=hourly["fraud_rate_pct"], ax=ax, color="#d62728")
ax.axhline(100 * n_fraud / total_txn, color="black", ls="--", label="Overall rate")
ax.set_title("Fraud rate by hour of day"); ax.set_xlabel("Hour"); ax.set_ylabel("% labelled fraud")
ax.legend(); plt.tight_layout(); plt.show()

fig, ax = plt.subplots(figsize=(12, 4.5))
ax.plot(monthly.index, monthly["fraud_rate_pct"], marker="o", color="#d62728")
ax.set_title("Monthly fraud rate"); ax.set_ylabel("% labelled fraud"); ax.set_xlabel("")
plt.tight_layout(); plt.show()

band = (df.groupby("amount_band", observed=True)
          .agg(txns=("amt", "size"), frauds=("is_fraud", "sum"))
          .assign(fraud_rate_pct=lambda x: 100 * x["frauds"] / x["txns"]))
display(band)

## 12. Customer (card) behaviour

In [ ]:
cust = (df.groupby("card_id", observed=True)
          .agg(txns=("amt", "size"), total_amt=("amt", "sum"), avg_amt=("amt", "mean"),
               frauds=("is_fraud", "sum")))
display(cust.describe().T)

cust_sorted = cust["total_amt"].sort_values(ascending=False)
top_n = max(1, int(len(cust) * 0.10))          # same rule as the SQL: FLOOR(10% of cards)
top10_share = 100 * cust_sorted.head(top_n).sum() / cust_sorted.sum()
cards_with_fraud_pct = 100 * (cust["frauds"] > 0).mean()
print(f"Top 10% of cards ({top_n} cards) account for {top10_share:.1f}% of total spend")
print(f"Cards with at least one fraud-labelled transaction: {cards_with_fraud_pct:.1f}%")

fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
sns.histplot(cust["total_amt"] / 1e3, bins=40, ax=ax[0], color="#9467bd")
ax[0].set_title("Total spend per card"); ax[0].set_xlabel("USD thousands")
sns.scatterplot(data=cust, x="txns", y="avg_amt", hue=(cust["frauds"] > 0), ax=ax[1], alpha=.6)
ax[1].set_title("Transactions vs average amount per card"); ax[1].legend(title="Has fraud label")
plt.tight_layout(); plt.show()

demo = (df.assign(age_band=pd.cut(df["customer_age"], [0, 25, 40, 55, 70, 120], right=False,
                                   labels=["<25", "25-39", "40-54", "55-69", "70+"]))
          .groupby(["gender", "age_band"], observed=True)
          .agg(txns=("amt", "size"), avg_amt=("amt", "mean"), frauds=("is_fraud", "sum"))
          .assign(fraud_rate_pct=lambda x: 100 * x["frauds"] / x["txns"]))
display(demo)

## 13. Outlier investigation
Outliers are checked with the IQR rule and the top 1% cut-off. Large amounts are **not removed**: they are valid transactions and often the ones that matter most.

In [ ]:
q1, q3 = df["amt"].quantile([.25, .75])
upper_fence = q3 + 1.5 * (q3 - q1)
p99 = df["amt"].quantile(.99)
iqr_out = df["amt"] > upper_fence
top1 = df["amt"] >= p99
print(f"IQR upper fence: {upper_fence:,.2f} -> {iqr_out.sum():,} transactions ({iqr_out.mean():.1%})")
print(f"99th percentile: {p99:,.2f} -> {top1.sum():,} transactions")

seg = pd.DataFrame({
    "txns": [top1.sum(), (~top1).sum()],
    "spend_share_pct": [100 * df.loc[top1, "amt"].sum() / total_amt, 100 * df.loc[~top1, "amt"].sum() / total_amt],
    "fraud_rate_pct": [100 * df.loc[top1, "is_fraud"].mean(), 100 * df.loc[~top1, "is_fraud"].mean()],
}, index=["Top 1% by amount", "Remaining 99%"])
display(seg)

fig, ax = plt.subplots(figsize=(10, 4.5))
sns.histplot(data=df[top1], x="amt", hue="fraud_label", bins=60, ax=ax,
             palette={"Legitimate": "#1f77b4", "Fraud": "#d62728"})
ax.set_title("Top 1% transactions by amount, split by label"); ax.set_xlabel("Amount (USD)")
plt.tight_layout(); plt.show()

## 14. Geography and distance

In [ ]:
st = (df.groupby("state", observed=True)
        .agg(txns=("amt", "size"), total_amt=("amt", "sum"), frauds=("is_fraud", "sum"))
        .assign(fraud_rate_pct=lambda x: 100 * x["frauds"] / x["txns"]))
min_vol = 10_000
st_big = st[st["txns"] >= min_vol].sort_values("fraud_rate_pct", ascending=False)
display(st.sort_values("total_amt", ascending=False).head(10))

fig, ax = plt.subplots(figsize=(10, 6))
sns.barplot(y=st_big.head(15).index.astype(str), x=st_big.head(15)["fraud_rate_pct"], ax=ax, color="#d62728")
ax.set_title(f"Top 15 states by fraud rate (states with >= {min_vol:,} transactions)")
ax.set_xlabel("% labelled fraud"); ax.set_ylabel("")
plt.tight_layout(); plt.show()

dist = df.groupby("fraud_label", observed=True)["distance_km"].describe()
display(dist)
fig, ax = plt.subplots(figsize=(9, 4.5))
sns.boxplot(data=sample, x="fraud_label", y="distance_km", ax=ax,
            palette=["#1f77b4", "#d62728"], showfliers=False)
ax.set_title("Customer-to-merchant distance by label (200k-row sample)")
ax.set_xlabel(""); ax.set_ylabel("km")
plt.tight_layout(); plt.show()

## 15. Export a lean, privacy-safer file for Power BI
Only the columns the dashboard needs are exported. **Not exported:** the card number, names, street, zip, city, coordinates, job and birth date. Customers appear only as `card_id`.
Save the CSV and download it from the Colab file panel (left sidebar, folder icon). Do not commit it to GitHub.

In [ ]:
export = pd.DataFrame({
    "trans_num": df["trans_num"],
    "trans_date_trans_time": df["trans_date_trans_time"],
    "txn_date": df["txn_date"].dt.strftime("%Y-%m-%d"),
    "txn_hour": df["txn_hour"],
    "card_id": df["card_id"],
    "merchant": df["merchant_clean"],
    "category": df["category"],
    "amt": df["amt"],
    "amount_band": df["amount_band"],
    "gender": df["gender"],
    "customer_age": df["customer_age"],
    "state": df["state"],
    "distance_km": df["distance_km"].round(2),
    "is_fraud": df["is_fraud"],
    "fraud_label": df["fraud_label"],
})
forbidden = {"cc_num", "first", "last", "street", "zip", "city", "lat", "long", "merch_lat", "merch_long", "job", "dob"}
assert not (forbidden & set(export.columns)), "Export contains a sensitive column"

out_file = "transactions_for_powerbi.csv"
export.to_csv(out_file, index=False)
print(f"Wrote {out_file} ({os.path.getsize(out_file)/1e6:,.0f} MB, {len(export):,} rows)")
print("Columns:", list(export.columns))
del export
# In Colab, uncomment to download directly:
# from google.colab import files; files.download(out_file)

## 16. Facts summary (computed, nothing typed by hand)
Copy this block into `Business_Insights/findings.md` after reviewing it. Every figure is calculated from the data above.

In [ ]:
top_cat_spend = cat.index[0]
hi_fraud_cat = fcat.index[0]
peak_hour = hourly["txns"].idxmax()
risk_hour = hourly["fraud_rate_pct"].idxmax()
fraud_med = df.loc[df["is_fraud"] == 1, "amt"].median()
legit_med = df.loc[df["is_fraud"] == 0, "amt"].median()

facts = f'''
ROWS: {total_txn:,} | PERIOD: {ts.min():%Y-%m-%d} to {ts.max():%Y-%m-%d}
CARDS: {df["card_id"].nunique():,} | MERCHANTS: {df["merchant_clean"].nunique():,} | CATEGORIES: {df["category"].nunique()}
TOTAL AMOUNT: {total_amt:,.2f} | AVG: {df["amt"].mean():.2f} | MEDIAN: {df["amt"].median():.2f} | P99: {p99:,.2f}
FRAUD: {n_fraud:,} txns = {100*n_fraud/total_txn:.3f}% of txns; {kpis["Fraud amount (% of total amount)"]:.2f}% of amount
MEDIAN AMT fraud={fraud_med:,.2f} vs legit={legit_med:,.2f}
TOP SPEND CATEGORY: {top_cat_spend} ({cat.loc[top_cat_spend, "pct_of_spend"]:.1f}% of spend)
HIGHEST FRAUD-RATE CATEGORY: {hi_fraud_cat} ({fcat.loc[hi_fraud_cat, "fraud_rate_pct"]:.2f}%)
BUSIEST HOUR: {peak_hour} | HIGHEST FRAUD-RATE HOUR: {risk_hour} ({hourly.loc[risk_hour, "fraud_rate_pct"]:.2f}%)
TOP 1% AMOUNT: {seg.loc["Top 1% by amount","spend_share_pct"]:.1f}% of spend, fraud rate {seg.loc["Top 1% by amount","fraud_rate_pct"]:.2f}% vs {seg.loc["Remaining 99%","fraud_rate_pct"]:.2f}%
TOP 10% CARDS ({top_n} cards): {top10_share:.1f}% of spend | CARDS WITH >=1 FRAUD LABEL: {cards_with_fraud_pct:.1f}%
'''
print(facts)
print("--- Category table ---");  print(cat[["txns","total_amt","pct_of_spend","fraud_rate_pct"]].round(2).to_string())
print("--- Hourly fraud rate ---"); print(hourly["fraud_rate_pct"].round(3).to_string())
print("--- Amount band ---");      print(band.round(3).to_string())
print("--- Monthly ---");          print(monthly[["txns","total_amt","fraud_rate_pct"]].round(2).to_string())
print("--- Top states by fraud rate ---"); print(st_big.head(5)[["txns","fraud_rate_pct"]].round(3).to_string())
print("--- Weekday ---");          print(weekday[["txns","total_amt","fraud_rate_pct"]].round(3).to_string())
print("--- Distance (km) by label ---"); print(dist[["mean","50%","max"]].round(1).to_string())
print("--- Age band x gender ---"); print(demo[["txns","avg_amt","fraud_rate_pct"]].round(3).to_string())